# 00. Train Model - Locally

Helper notebook for training fire risk model locally.

Recommended for making sure model + data are correct. Training full model locally is time consuming

IMPORTANT: Currently, multi-dir training shuffles features and labels so that they are mismatched
Need to fix either in geebeam export, in an offline post-processing step, or at training time by joining
based on index.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
import aic_risk_modeling as arm
import matplotlib.pyplot as plt


In [ ]:
# mcwd_stats = arm.train.load_stats_from_text('gs://aic-fire-amazon/data/terraclim_mcwd/stats.pbtxt')
# arm.train.get_norm_stats(mcwd_stats, 'mcwd_2022')
# veg_stats = arm.train.load_stats_from_text('gs://aic-fire-amazon/data/allpreds/stats.pbtxt')
# arm.train.get_norm_stats(veg_stats, 'EVI_begyear_2021')

In [ ]:
SEED = 54
RNG = np.random.default_rng(SEED)

# Set params

In [ ]:
DATA_DIRS = ["../../data/mb_amz_forest/","../../data/mcd64_burndate/", "../../data/terraclim_mcwd/", '../../data/modis_veg/']
TFRECORD_PATTERN='*.tfrecord.gz'
PATCH_SIZE=128

# For model, input and output bands
INPUT_BANDS = (
    "BurnDate", "classification", "mcwd", "EVI", "NDVI"
)
OUTPUT_BANDS = ['BurnDate_2024']

YEARS = [2021, 2022, 2023]

TRANSFORMS = {
        "BurnDate": "gt0",
        "BurnDate_2024": "gt0_bool",
        "mcwd": "normalize_mcwd",
        "EVI": "normalize_evi",
        "NDVI": "normalize_ndvi",
    }

In [ ]:
def build_merged_dataset(
        data_dirs,
        tfrecord_pattern,
        patch_size,
        batch_size=4
        ):
    training_datasets = []
    training_pattern = 'training-{}'.format(tfrecord_pattern)
    validation_datasets = []
    validation_pattern = 'validation-{}'.format(tfrecord_pattern)
    for data_dir in data_dirs:
        print(data_dir)
        training_ds = arm.train.data_loader.dataset_from_dir(
            data_dir,
            training_pattern,
            patch_size=patch_size,
            batch_size=batch_size,
            cache=False
        )
        validation_ds = arm.train.data_loader.dataset_from_dir(
            data_dir,
            validation_pattern,
            patch_size=patch_size,
            batch_size=batch_size,
            cache=False
        )
        training_datasets.append(training_ds)
        validation_datasets.append(validation_ds)

    training_merged = arm.train.data_loader.merge_datasets(training_datasets).shuffle(buffer_size=64)
    validation_merged = arm.train.data_loader.merge_datasets(validation_datasets)

    return training_merged, validation_merged

In [ ]:

# Get datasets
training_ds, validation_ds = build_merged_dataset(
    data_dirs=DATA_DIRS,
    tfrecord_pattern=TFRECORD_PATTERN,
    patch_size=PATCH_SIZE,
    batch_size=4,
)

# Select bands
training_ds = arm.train.data_loader.select_bands_transform(
    training_ds,
    input_bands=INPUT_BANDS,
    output_bands=OUTPUT_BANDS,
    transforms=TRANSFORMS,
    stack_time_series=True,
    stack_inputs=False,
    years=YEARS
)
validation_ds = arm.train.data_loader.select_bands_transform(
    validation_ds,
    input_bands=INPUT_BANDS,
    output_bands=OUTPUT_BANDS,
    transforms=TRANSFORMS,
    stack_time_series=True,
    stack_inputs=False,
    years=YEARS
)

In [ ]:
for inputs, labels in training_ds.take(1):
    print("Batch inputs keys:", list(inputs.keys()))
    print("Image shape:", inputs['image'].shape)
    print("Label shape:", labels.shape)
    print("Label dtype", labels.dtype)
    print("Input dtype", inputs['image'].dtype)
    input_shape = inputs['image'].shape[1:] 
    plt.imshow(inputs['image'][0,1, :, :, 0])
    plt.show()
    plt.imshow(inputs['image'][0,1, :, :, 1])
    plt.show()
    plt.imshow(inputs['image'][0,1, :, :, 2])
    plt.show()
    plt.imshow(inputs['image'][0,1, :, :, 3])
    plt.show()
    plt.imshow(inputs['image'][0,1, :, :, 4])
    plt.show()
    plt.imshow(labels[0])

In [ ]:
model = arm.train.get_simple_convlstm(input_shape=input_shape)
model.summary()

In [ ]:
# Define the input dictionary layers.
image_input = tf.keras.Input(shape=input_shape, name='image')
new_model = tf.keras.Model(image_input, model(image_input))

In [ ]:
new_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0025),
    loss="Dice",
    metrics=[
        tf.keras.metrics.BinaryIoU(target_class_ids=[1]),
        tf.keras.metrics.AUC(),
    ]
    )

checkpoint_filepath = './checkpoint.model.keras'
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_loss',
    mode='min',
    save_best_only=True)

early_stopping_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=30)

new_model.fit(
    training_ds,
    validation_data=validation_ds,
    epochs=25,
    callbacks=[model_checkpoint_callback, early_stopping_callback]
)

In [ ]:
new_model = tf.keras.models.load_model('checkpoint.model.keras')

In [ ]:
valid_masks = np.array([b[1][i].numpy() for b in training_ds for i in range(b[1].shape[0])])

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, jaccard_score

In [ ]:
out = new_model.predict(training_ds)

In [ ]:

print(f1_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(recall_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(precision_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(jaccard_score(valid_masks.flatten()>0.5, out.flatten()>0.5))

In [ ]:
def visualize_risk_predict(input_batch, target_batch, output_i, batch_i, suptitle, cutoff=0.5):
    fig, axs = plt.subplots(1,3)
    fig.suptitle(suptitle)
    # Embeddings
    rgb = np.stack([
        input_batch['image'][batch_i, 2, :, :, 1].numpy(),
        input_batch['image'][batch_i, 2, :, :, 16].numpy(),
        input_batch['image'][batch_i, 2,:, :, 9].numpy()], axis=2)
    # shift
    vmin=-0.3
    vmax=0.3
    rgb = (rgb - vmin)/(vmax - vmin)
    axs.flatten()[0].imshow(rgb)
    axs.flatten()[0].set_title('Embeddings')


    # Prediction
    axs.flatten()[1].imshow(output_i)
    axs.flatten()[1].set_title('Predicted burned area 2024')

    # 2023 burn (target)
    axs.flatten()[2].imshow(target_batch[batch_i].numpy()>cutoff)
    axs.flatten()[2].set_title('Actual burned area 2024')
    fig.tight_layout()

    plt.show()


In [ ]:
import matplotlib.pyplot as plt
j = 0
for batch in training_ds:
    for i in range(batch[1].shape[0]):
        if (batch[1][i].numpy()>0.5).sum()>0 or (out[j]>0.5).sum()>0:
            visualize_risk_predict(
                input_batch = batch[0],
                target_batch = batch[1],
                output_i = out[j],
                batch_i=i,
                suptitle='Image {}'.format(j),
                cutoff=0.99
            )
        j+=1